# MLOps Pipeline: Cats vs Dogs Classification
This notebook demonstrates the end-to-end process of downloading the dataset, preprocessing it, defining a model, and training it with MLflow tracking.

## 1. Setup and Installation
Ensure we have the required libraries installed. (Uncomment and run if needed)

In [ ]:
!pip install -r ../requirements.txt

## 2. Dataset Download
We will download the dataset from Kaggle using the Kaggle API. 
Make sure your `kaggle.json` is properly configured in `~/.kaggle/kaggle.json`.

In [ ]:
import os
import subprocess
import shutil

def download_kaggle_dataset(dataset_name="salader/dogs-vs-cats", download_path="../data/raw"):
    print(f"Downloading {dataset_name} to {download_path}...")
    os.makedirs(download_path, exist_ok=True)
    
    subprocess.run([
        "kaggle", "datasets", "download", 
        "-d", dataset_name, 
        "-p", download_path, 
        "--unzip"
    ], check=True)
    
    cat_dir = os.path.join(download_path, "Cat")
    dog_dir = os.path.join(download_path, "Dog")
    os.makedirs(cat_dir, exist_ok=True)
    os.makedirs(dog_dir, exist_ok=True)
    
    for split in ['train', 'test']:
        split_dir = os.path.join(download_path, split)
        if not os.path.exists(split_dir):
            continue
            
        for class_name in ['cats', 'dogs']:
            src_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(src_dir):
                continue
                
            dest_dir = cat_dir if class_name == 'cats' else dog_dir
            for f in os.listdir(src_dir):
                if f.endswith('.jpg'):
                    new_name = f"{split}_{f}"
                    shutil.move(os.path.join(src_dir, f), os.path.join(dest_dir, new_name))
                    
        shutil.rmtree(split_dir)
    print("Download and reorganization complete.")

# Uncomment the line below to download data (takes some time)
# download_kaggle_dataset()

## 3. Data Preprocessing
We define a custom PyTorch dataset to handle the images, apply augmentations, and load them into DataLoaders.

In [ ]:
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

class CatsDogsDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor([label], dtype=torch.float32)

def get_dataloaders(raw_data_dir='../data/raw', batch_size=32):
    cat_files = glob.glob(os.path.join(raw_data_dir, 'Cat', '*.jpg'))
    dog_files = glob.glob(os.path.join(raw_data_dir, 'Dog', '*.jpg'))
    
    all_files = cat_files + dog_files
    labels = [0]*len(cat_files) + [1]*len(dog_files) # 0 for Cat, 1 for Dog
    
    # Stratified Split (80% Train, 10% Val, 10% Test)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        all_files, labels, test_size=0.1, random_state=42, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.1111, random_state=42, stratify=y_train_val 
    )
    
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(CatsDogsDataset(X_train, y_train, train_transform), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(CatsDogsDataset(X_val, y_val, val_test_transform), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(CatsDogsDataset(X_test, y_test, val_test_transform), batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# Create dataloaders
# train_loader, val_loader, test_loader = get_dataloaders()
# print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 4. Model Architecture
Defining a simple Convolutional Neural Network (CNN) as our baseline model.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(64 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 1) 
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()
print(model)

## 5. Training with MLflow Tracking
Training the model and using MLflow to track hyperparameters and loss/accuracy metrics.

In [ ]:
import torch.optim as optim
import mlflow

def train_model(model, train_loader, val_loader, epochs=5, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    mlflow.set_experiment("Cats_vs_Dogs_Notebook_Experiment")
    with mlflow.start_run():
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("learning_rate", lr)
        
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            
            # Using a tiny fraction of data just to demonstrate if needed
            # for inputs, labels in train_loader:
            #     inputs, labels = inputs.to(device), labels.to(device)
            #     optimizer.zero_grad()
            #     outputs = model(inputs)
            #     loss = criterion(outputs, labels)
            #     loss.backward()
            #     optimizer.step()
            #     running_loss += loss.item() * inputs.size(0)
                
            # epoch_loss = running_loss / len(train_loader.dataset)
            epoch_loss = 0.0 # Dummy
            
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")
            mlflow.log_metric("train_loss", epoch_loss, step=epoch)
            
        torch.save(model.state_dict(), "../model.pt")
        mlflow.log_artifact("../model.pt")
        print("Training completed and model saved!")

# Run training
# train_model(model, train_loader, val_loader, epochs=1)

## 6. Inference / Packaging Next Steps
Now that the model is trained (`../model.pt`), the MLOps pipeline takes over to containerize and serve it via FastAPI, orchestrated by Docker Compose. You can run `docker-compose up --build -d` from the terminal to spin up the REST API and Streamlit UI!